# 11 — Trend Alignment

## Goal
Score whether a zone trades **with** or **against** the current price trend.  
A demand zone in an uptrend is far more likely to hold than one fighting the trend.

---

## Swing Structure

We define trend via **swing highs and swing lows** over a rolling window of `SWING_WINDOW` candles:

$$\text{swing\_high}[i] = \text{high}[i] = \max(\text{high}[i - w \,\ldots\, i + w])$$
$$\text{swing\_low}[i]  = \text{low}[i]  = \min(\text{low}[i - w \,\ldots\, i + w])$$

where $w = \text{SWING\_WINDOW} = 3$.

Trend is classified at the base start (`bs`):

| Condition | Trend |
|-----------|-------|
| Last swing high **>** previous swing high AND last swing low **>** previous swing low | **uptrend** |
| Last swing high **<** previous swing high AND last swing low **<** previous swing low | **downtrend** |
| Otherwise | **sideways** |

---

## Scoring

| Zone type | Trend | Score |
|-----------|-------|-------|
| demand    | uptrend   | 2 |
| supply    | downtrend | 2 |
| any       | sideways  | 1 |
| demand    | downtrend | 0 |
| supply    | uptrend   | 0 |

---

## Visualization
Swing highs/lows plotted on the chart; zones coloured by alignment score.


## 1. Load data and run the detection pipeline

In [1]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


9 zones detected


## 2. Swing structure and trend at each bar

In [2]:
SWING_WINDOW = 3

def find_swings():
    sh_idx, sl_idx = [], []
    for i in range(SWING_WINDOW, len(h_arr) - SWING_WINDOW):
        if h_arr[i] == h_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].max():
            sh_idx.append(i)
        if l_arr[i] == l_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].min():
            sl_idx.append(i)
    return sh_idx, sl_idx

sh_idx, sl_idx = find_swings()
print(f"Swing highs: {len(sh_idx)}   Swing lows: {len(sl_idx)}")

def trend_at(pos):
    past_sh = [i for i in sh_idx if i < pos]
    past_sl = [i for i in sl_idx if i < pos]
    if len(past_sh) < 2 or len(past_sl) < 2:
        return "sideways"
    hh = h_arr[past_sh[-1]] > h_arr[past_sh[-2]]
    hl = l_arr[past_sl[-1]] > l_arr[past_sl[-2]]
    lh = h_arr[past_sh[-1]] < h_arr[past_sh[-2]]
    ll = l_arr[past_sl[-1]] < l_arr[past_sl[-2]]
    if hh and hl: return "uptrend"
    if lh and ll: return "downtrend"
    return "sideways"

TREND_SCORE = {
    ("demand", "uptrend"):   2, ("demand", "sideways"):  1, ("demand", "downtrend"): 0,
    ("supply", "downtrend"): 2, ("supply", "sideways"):  1, ("supply", "uptrend"):   0,
}

for z in zones:
    z["trend"]        = trend_at(z["bs"])
    z["trend_score"]  = TREND_SCORE.get((z["zone_type"], z["trend"]), 1)
    z["scenario"]     = labeled["scenario"].iloc[z["bs"]]

print(f"\n{'Scenario':<22} {'Type':8} {'Trend':10} {'Score':>5}")
print("-" * 50)
for z in zones:
    print(f"{z['scenario']:<22} {z['zone_type']:8} {z['trend']:10} {z['trend_score']:>5}")


Swing highs: 6   Swing lows: 8

Scenario               Type     Trend      Score
--------------------------------------------------
A_demand_RBR           demand   sideways       1
A_demand_RBR           supply   sideways       1
B_supply_DBD           supply   sideways       1
D_wide_base            demand   sideways       1
E_doji_base            demand   sideways       1
E_doji_base            demand   sideways       1
H_DBR                  demand   uptrend        2
H_DBR                  demand   sideways       1
I_RBD                  supply   sideways       1


## 3. Visualization — swing structure and trend scores

In [3]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

SCORE_COLOR = {0: "#ef5350", 1: "#ffb74d", 2: "#26a69a"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

fig.add_trace(go.Scatter(
    x=df.index[sh_idx], y=h_arr[sh_idx],
    mode="markers", marker=dict(symbol="triangle-up", size=8, color="#26a69a"),
    name="Swing High",
))
fig.add_trace(go.Scatter(
    x=df.index[sl_idx], y=l_arr[sl_idx],
    mode="markers", marker=dict(symbol="triangle-down", size=8, color="#ef5350"),
    name="Swing Low",
))

for z in zones:
    x0, x1 = df.index[z["bs"]], df.index[z["be"]]
    c_edge  = SCORE_COLOR[z["trend_score"]]
    fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(255,255,255,0.05)",
                  line=dict(color=c_edge, width=1.5, dash="dot"))
    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']} align={z['trend_score']}",
                       showarrow=False, font=dict(size=8, color=c_edge),
                       xanchor="left", yanchor="bottom")

fig.update_layout(
    title="Trend Alignment Score per Zone",
    height=540, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
